# Latent Language Lab — Colab
Train independent English and Spanish autoencoders, test decoder swaps, then fit a frozen-model alignment map.

**Before running:** choose **Runtime → Change runtime type → GPU**, selecting A100 when available. Upload this project to GitHub first, then paste its public repository URL below. No GitHub token is needed for a public repository.

This notebook trains small models from scratch. Results are experimental, not a ready-made translation system. Checkpoints go to your Google Drive; source code runs in Colab's temporary filesystem.

In [1]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/joyc2point718/latent-language.git"
PROJECT = Path("/content/latent-language-lab")
if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Replace REPO_URL with your GitHub repository URL first.")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
print("Working directory:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Working directory: /content/latent-language-lab
Commit: 12af5eea9f45b28ab874494e8b8f89810c9d8390


## Install the small dependency set
Keep Colab's existing GPU-enabled PyTorch installation. These commands install the additional libraries and make `latentlab` importable. Rerunning this cell is safe. If Colab requests a restart, restart and rerun the setup cells.

In [2]:
%pip install -q -r requirements.txt
%pip install -q -e . --no-deps

import torch
from packaging.version import Version
assert Version("2.6") <= Version(torch.__version__.split("+")[0]) < Version("3"), "Requires PyTorch >=2.6,<3"
assert torch.cuda.is_available(), "Choose a GPU under Runtime → Change runtime type."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for latent-language-lab (pyproject.toml) ... done
PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


In [3]:
from pathlib import Path
import os

os.chdir("/content/latent-language-lab")

for name in ["requirements.txt", "pyproject.toml"]:
    path = Path(name)
    path.write_text(
        path.read_text().replace("tokenizers==0.22.2", "tokenizers==0.23.1")
    )

%pip install -r requirements.txt
%pip install -e . --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
latent-language-lab 0.1.0 requires tokenizers==0.22.2, but you have tokenizers 0.23.1 which is incompatible.
Obtaining file:///content/latent-language-lab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for latent-language-lab (pyproject.toml) ... done
  Created wheel for latent-language-lab: filename=latent_language_lab-0.1.0-0.editable-py3-none-any.whl size=3191 sha256=52f8c3faa22b0e8a3e9fad

In [4]:
!python -m unittest discover -s tests -v

test_coordinate_recovery (test_experiment.ExperimentTests.test_coordinate_recovery) ... ok
test_data_separation (test_experiment.ExperimentTests.test_data_separation) ... ok
[00:00:00] Tokenize words                 ██████████████████ 34       /       34
[00:00:00] Count pairs                    ██████████████████ 34       /       34
[00:00:00] Compute merges                 ██████████████████ 89       /       89
{"parameters": 48448, "device": "cpu", "bf16": false, "start_epoch": 0}
epoch=1 batch=0 loss=30.9470
{"epoch": 1, "train_loss": 30.377442103165848, "val_masked_loss": 30.477736253004807}
[00:00:00] Tokenize words                 ██████████████████ 29       /       29
[00:00:00] Count pairs                    ██████████████████ 29       /       29
[00:00:00] Compute merges                 ██████████████████ 97       /       97
{"parameters": 48704, "device": "cpu", "bf16": false, "start_epoch": 0}
epoch=1 batch=0 loss=30.6453
{"epoch": 1, "train_loss": 30.478390719095867, "val_

## Mount Drive and record the environment
Colab will ask you to authorize Drive access. Choose a new `EXPERIMENT` name for a new dataset or architecture. Rerunning an existing experiment resumes completed epochs where possible.

Default checkpoint storage is about 560 MB for both best and latest checkpoints across all three models, plus data/results. Code belongs in GitHub; these working checkpoints can stay in Drive. Hugging Face is optional for sharing completed models.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

EXPERIMENT = "controlled-v1"
ROOT = Path("/content/drive/MyDrive/latent-language-lab") / EXPERIMENT
DATA = ROOT / "data"
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
(ROOT / f"requirements-resolved-{stamp}.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(ROOT / f"gpu-{stamp}.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(ROOT / f"commit-{stamp}.txt").write_text(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))

def run(*args):
    subprocess.run([sys.executable, "-m", "latentlab.cli", *map(str,args)], check=True)

print("Persistent experiment directory:", ROOT)

## Generate the controlled dataset
The default corpus has 20,000 unique training events, separate validation/alignment/test sets, and a held-out-combination set. English and Spanish training files are shuffled independently. Pairing is used only for evaluation and the explicitly supervised alignment stage.

In [ ]:
if not (DATA / "manifest.json").exists():
    run("prepare", "--out", DATA)
print((DATA / "manifest.json").read_text())

## Train the first English model
Start with the defaults. Increase `EPOCHS` later to continue training; it is the total desired number of epochs, not additional epochs. A completed epoch writes `last.pt`; `best.pt` is selected by masked validation loss. An interrupted epoch restarts from the previous completed epoch.

The initial architecture is 4 encoder + 4 decoder layers, width 256, 8 latent slots. Training uses BF16 when supported. Do not change architecture or training hyperparameters within an existing run.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 128

def train_run(name, lang, seed, tokenizer=None):
    output = RUNS / name
    args = ["train", "--data", DATA, "--lang", lang, "--seed", seed,
            "--out", output, "--epochs", EPOCHS, "--batch-size", BATCH_SIZE]
    if tokenizer:
        args += ["--tokenizer", tokenizer]
    if (output / "last.pt").exists():
        args += ["--resume"]
    run(*args)

train_run("en_a", "en", 1)

## Check reconstruction before interpreting swaps
This cell uses validation data. Compare normal reconstruction with incorrect latent conditioning. View the actual predictions as well as the scores. If reconstruction is poor, train longer or debug before drawing conclusions from swaps.

In [ ]:
EN_A = RUNS / "en_a/best.pt"
run("evaluate", "--source", EN_A, "--target", EN_A,
    "--pairs", DATA / "val.jsonl", "--limit", 100, "--out", RUNS / "en_reconstruction_val.json")
run("evaluate", "--source", EN_A, "--target", EN_A,
    "--pairs", DATA / "val.jsonl", "--limit", 100, "--shuffle-latents",
    "--out", RUNS / "en_shuffled_val.json")
import json
examples = (RUNS / "en_reconstruction_val.predictions.jsonl").read_text().splitlines()
for line in examples[:5]:
    print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))

## Train English seed B and Spanish
The second English model reuses the first English model's tokenizer, but starts with independent model weights. Spanish trains its own tokenizer. These runs train independently and sequentially on the same GPU.

In [ ]:
train_run("en_b", "en", 2, RUNS / "en_a/tokenizer.json")
train_run("es_a", "es", 3)
EN_B = RUNS / "en_b/best.pt"
ES_A = RUNS / "es_a/best.pt"
for name, checkpoint in [("en_b", EN_B), ("es_a", ES_A)]:
    run("evaluate", "--source", checkpoint, "--target", checkpoint,
        "--pairs", DATA / "val.jsonl", "--limit", 100,
        "--out", RUNS / f"{name}_reconstruction_val.json")

## Fit alignment on its reserved split
No autoencoder weights change here. Each map is a shared feature-space transformation across slots; corresponding slot indices are assumed. A failed map may reflect incompatible slot organization. The shuffled-pair bridge is a negative control. Fit only on `align.jsonl`, never the final test set.

In [ ]:
for label, target in [("en_en", EN_B), ("en_es", ES_A)]:
    for kind in ["orthogonal", "linear"]:
        run("align", "--source", EN_A, "--target", target,
            "--pairs", DATA / "align.jsonl", "--n", 1000, "--kind", kind,
            "--out", RUNS / f"{label}_{kind}.pt")
run("align", "--source", EN_A, "--target", ES_A,
    "--pairs", DATA / "align.jsonl", "--n", 1000, "--shuffle-pairs",
    "--out", RUNS / "en_es_shuffled_pairs.pt")

## Final comparisons
Run after settling training decisions using validation. Evaluate the held-out test and compositional split. Exact match is strict string equality; chrF is a surface similarity metric. Neither alone establishes shared semantics. Inspect role reversals, negation and verb errors in the prediction files.

In [ ]:
conditions = [
    ("en_reconstruction", EN_A, EN_A, None),
    ("es_reconstruction", ES_A, ES_A, None),
    ("english_swap", EN_A, EN_B, None),
    ("spanish_swap", EN_A, ES_A, None),
    ("english_orthogonal", EN_A, EN_B, RUNS / "en_en_orthogonal.pt"),
    ("english_linear", EN_A, EN_B, RUNS / "en_en_linear.pt"),
    ("spanish_orthogonal", EN_A, ES_A, RUNS / "en_es_orthogonal.pt"),
    ("spanish_linear", EN_A, ES_A, RUNS / "en_es_linear.pt"),
    ("spanish_shuffled_pairs", EN_A, ES_A, RUNS / "en_es_shuffled_pairs.pt"),
]
for split in ["test", "ood"]:
    for name, source, target, bridge in conditions:
        args = ["evaluate", "--source", source, "--target", target,
                "--pairs", DATA / f"{split}.jsonl", "--out", RUNS / f"{name}_{split}.json"]
        if bridge:
            args += ["--bridge", bridge]
        run(*args)

In [ ]:
import csv
rows = []
for split in ["test", "ood"]:
    for name, *_ in conditions:
        result = json.loads((RUNS / f"{name}_{split}.json").read_text())
        rows.append({"split":split,"condition":name,"n":result["n"],
                     "chrF":result["chrf"],"exact_match":result["exact_match"]})
with (RUNS / "comparison.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
from tabulate import tabulate
print(tabulate(rows, headers="keys", floatfmt=".3f"))
print("Results and predictions:", RUNS)

## Use the models directly from Python
The source encoder and target decoder can be attached without retraining. Their latent shapes must agree. Direct swaps may produce nonsense; that is an experimental outcome, not necessarily a software bug.

In [ ]:
from latentlab.experiment import load_model, apply_map
from latentlab.data import encode_texts, padded

source, tokenizer_en, _ = load_model(EN_A, "cuda")
target, tokenizer_es, _ = load_model(ES_A, "cuda")
sentence = "The red dog follows the blue cat."
ids = padded(encode_texts(tokenizer_en, [sentence], source.config.max_len), "cuda")
with torch.no_grad():
    z = source.encode(ids)
    direct = target.generate(z)
    bridge = torch.load(RUNS / "en_es_orthogonal.pt", map_location="cpu", weights_only=True)
    aligned = target.generate(apply_map(z, bridge))
print("Input:", sentence)
print("Direct:", tokenizer_es.decode(direct[0].tolist(), skip_special_tokens=True))
print("Aligned:", tokenizer_es.decode(aligned[0].tolist(), skip_special_tokens=True))

## Next experiments
- Repeat key conditions across multiple seeds.
- Compare 1, 8 and 16 latent slots with separate output directories.
- Fit on 100 vs 1,000 alignment pairs; hold evaluation fixed.
- Try Spanish → English by reversing source and target and fitting a new map.
- Move to natural text only after reliable held-out reconstruction. See the README for the input file format and OPUS-100 guidance.

This notebook does not automatically download natural corpora, train audio models, or implement EEG decoding.